# thay_label_run — SINH NHÃN TỪ MODEL THẦY (Qwen3-Reranker-8B) CHO CHƯNG CẤT

**BTC đã xác nhận (25/08): chưng cất từ model vượt trần là HỢP LỆ**, vì khi nộp bài chỉ có
model học trò ≤3B chạy. Notebook này **CHỈ sinh nhãn**. Nó **không** tạo file `submission`
nào — có một `assert` ở cuối kiểm đúng điều đó.

### Vì sao đáng làm — con số đo được, không phải hy vọng

Trên 15 câu khó, **cùng một rổ ứng viên**:

| bộ chấm | R@1 | R@5 | R@10 |
|---|---|---|---|
| `ce_ours` (AITeamVN 0,568B) | 0,333 | 0,400 | 0,533 |
| Qwen3-Reranker-**4B** | 0,267 | 0,467 | 0,733 |
| **Qwen3-Reranker-8B** | 0,267 | **0,667** | **0,867** |

Và trên 11 câu dev300 mà pipeline **đang sai**: `ce_ours` 2/11 → **8B 6/11**.
→ Thầy biết thêm ~4 câu/11. Suy ra cả 18 câu sai thì trần khoảng **+2,0 điểm dev300**.
**4B gần như không hơn `ce_ours` → phải dùng 8B, không dùng 4B.**

### Vì sao chưng cất khác hai lần fine-tune đã thua

Hai lần trước dùng **nhãn cứng** (gold = 1, còn lại = 0) trên negative BM25 khó, và đã ghi
nhận nguyên nhân *"negative quá khó"*: nhiều "negative" thật ra có liên quan, phạt chúng
xuống 0 là **dạy sai**. Nhãn mềm của thầy không có lỗi đó — negative liên quan được thầy
cho điểm cao, học trò không bị phạt vì xếp nó cao. **Đây là lý do cơ chế để tin lần này khác.**

### Chi phí — đọc kỹ trước khi bấm

Thầy chạy **4,69 giây/cặp** (đo thật 23/08, đã batch 8, 4-bit trên T4).
Cấu hình mặc định `N_CAU=600` × 8 ứng viên = **4.800 cặp ≈ 6,3 giờ**.
Có checkpoint: hết giờ thì upload `outputs/` lên dataset, chạy lại là **chạy tiếp**, không mất gì.



In [ ]:
!pip install -q -U bitsandbytes accelerate



In [ ]:
# ===== Bước 0: KIỂM ĐỦ FILE TRƯỚC. Thiếu thì liệt kê HẾT, đừng chết từng cái một =====
import os, glob, hashlib
INPUT_DIR = next((p for p in ("/kaggle/input/project-ir",
                              "/kaggle/input/datasets/locdovan211/project-ir")
                  if os.path.isdir(p)), None)
assert INPUT_DIR, "không thấy dataset project-ir"

# tìm ĐỆ QUY: file có thể ở gốc dataset hoặc trong thư mục con, đừng đoán cây thư mục
can = ["deep_chunk.py", "rerank_qwen.py", "rerank_from_d.py",
       "train.json", "bm25_ids_train_FALLBACK.json"]
thieu = [f for f in can if not glob.glob(f"{INPUT_DIR}/**/{f}", recursive=True)]
ctx = glob.glob(f"{INPUT_DIR}/**/context_*.json", recursive=True)
if not ctx: thieu.append("selected-contexts/context_*.json")
assert not thieu, "⛔ THIẾU, upload rồi chạy lại:\n  - " + "\n  - ".join(thieu)
print(f"đủ {len(can)} file · {len(ctx):,} context")

# rerank_qwen.py PHẢI là bản có cửa thay_offline, nếu không nó chặn thẳng model 8B
src = open(glob.glob(f"{INPUT_DIR}/**/rerank_qwen.py", recursive=True)[0], encoding="utf-8").read()
assert "thay_offline" in src, (
    "⛔ rerank_qwen.py trong dataset là BẢN CŨ — không có cửa `thay_offline`, "
    "nó sẽ chặn model 8B ở trần 3B. Upload đè bản mới trong project_DSC.")
assert "BUDGET = 3_000_000_000" in src, "⛔ rerank_qwen.py bị nới trần — KHÔNG được, sửa lại"
print(f"rerank_qwen.py: có cửa thay_offline ✅ · trần 3B còn nguyên ✅ · "
      f"sha {hashlib.sha256(src.encode()).hexdigest()[:12]}")



import torch
ng = torch.cuda.device_count()
print(f"GPU: {ng} × {torch.cuda.get_device_name(0) if ng else '—'}")
assert ng >= 1, "CẦN GPU"
if ng == 1:
    print("⚠️  CHỈ 1 GPU. Thầy 8B nạp vừa nhưng RẤT sát trần — bản transformers mới đã OOM\n"
          "    ở đúng chỗ này. Nên chọn Accelerator = GPU T4 x2 (trải qua 2 card, 29GB).")



In [ ]:
# ===== Bước 1: dựng nhóm ứng viên =====
import os, sys, json, time, random, glob
import torch

N_CAU     = 600          # số câu train lấy nhãn. Tăng khi lượt thí điểm cho tín hiệu.
K_UNGVIEN = 8            # 1 gold + 7 negative khó (BM25). Nhóm để chưng cất theo LISTWISE.
EXC       = 900          # ký tự trích — PHẢI GIỐNG hệt lúc suy luận, nếu không nhãn lệch miền
GIO_TRAN  = 9.0          # dừng sạch trước trần 12h của Kaggle
SEED      = 20260825

INPUT_DIR = next(p for p in ("/kaggle/input/project-ir",
                             "/kaggle/input/datasets/locdovan211/project-ir")
                 if os.path.isdir(p))
CTX_DIR = next(p for p in (f"{INPUT_DIR}/selected-contexts/selected-contexts",
                           f"{INPUT_DIR}/selected-contexts")
               if os.path.isdir(p) and any(f.startswith("context_") for f in os.listdir(p)))
OUT = "/kaggle/working/outputs"; os.makedirs(OUT, exist_ok=True)
sys.path.append(INPUT_DIR)

def tim(ten):
    """Tìm file trong dataset dù nó nằm ở gốc hay trong thư mục con.
    BẪY 25/08: hardcode "Input/train.json" theo cây thư mục máy cá nhân, nhưng trên Kaggle
    file được upload ra GỐC dataset -> FileNotFoundError sau 22 giây. Đừng đoán cây thư mục."""
    p = glob.glob(f"{INPUT_DIR}/**/{ten}", recursive=True)
    assert p, f"KHÔNG THẤY {ten} trong dataset — upload rồi chạy lại"
    return sorted(p, key=len)[0]

import deep_chunk as DC
DC.MERGE_CHARS = 1800                      # BẪY CŨ: mặc định của module là 0 (chunk thô)
assert DC.MERGE_CHARS == 1800

train = json.load(open(tim("train.json"), encoding="utf-8"))
bm25  = json.load(open(tim("bm25_ids_train_FALLBACK.json"), encoding="utf-8"))
co_ctx = lambda d: os.path.isfile(f"{CTX_DIR}/context_{d}.json")

random.seed(SEED)
qs_all = sorted(q for q in train if q in bm25)
random.shuffle(qs_all)

nhom = {}                                   # {qid: [doc_id, ...]} — phần tử ĐẦU luôn là gold
for q in qs_all:
    if len(nhom) >= N_CAU: break
    gold = [str(a) for a in train[q]["answer"] if co_ctx(str(a))]
    if not gold: continue
    neg = [str(c["doc_id"]) for c in bm25[q]
           if str(c["doc_id"]) not in gold and co_ctx(str(c["doc_id"]))]
    if len(neg) < K_UNGVIEN - 1: continue
    nhom[q] = gold[:1] + neg[:K_UNGVIEN - 1]

print(f"{len(nhom)} câu × {K_UNGVIEN} = {len(nhom)*K_UNGVIEN:,} cặp")
print(f"ước tính: {len(nhom)*K_UNGVIEN*4.69/3600:.1f} giờ ở 4,69 s/cặp")
assert len(nhom) == N_CAU, f"chỉ dựng được {len(nhom)}/{N_CAU} câu"



In [ ]:
# ===== Bước 2: trích đoạn — ĐÚNG hàm dùng lúc suy luận =====
t0 = time.time()
cap, chiso = [], []
for i, (q, docs) in enumerate(nhom.items(), 1):
    qs = train[q]["question"]
    for d in docs:
        e = DC.pick_chunks(qs, CTX_DIR, d, k=1)
        cap.append([qs, (e[0] if e else "")[:EXC]]); chiso.append((q, d))
    if i % 200 == 0: print(f"  băm {i}/{len(nhom)} | {(time.time()-t0)/60:.1f} phút", flush=True)
print(f"\n{len(cap):,} cặp · {(time.time()-t0)/60:.1f} phút")
assert len(cap) == len(nhom) * K_UNGVIEN
rong = sum(1 for _, t in cap if not t.strip())
print(f"đoạn rỗng: {rong}" + ("  <-- KIỂM LẠI" if rong else "  ✅"))



In [ ]:
# ===== Bước 3: thầy chấm. Có checkpoint, hết giờ là chạy tiếp được =====
from rerank_qwen import load_qwen_reranker

THAY = "Qwen/Qwen3-Reranker-8B"
P_OUT = f"{OUT}/nhan_thay.json"

# nạp lại phần đã chấm (lượt trước upload outputs/ lên dataset)
cu = glob.glob(f"{INPUT_DIR}/**/nhan_thay.json", recursive=True)
nhan = json.load(open(P_OUT if os.path.isfile(P_OUT) else cu[0], encoding="utf-8")) \
       if (os.path.isfile(P_OUT) or cu) else {}
xong = {(q, d) for q, e in nhan.items() for d in e}
con = [i for i, k in enumerate(chiso) if k not in xong]
print(f"đã có {len(xong):,} cặp · còn {len(con):,} cặp")

if con:
    # SẮP THEO ĐỘ DÀI: batch toàn câu dài-ngắn lẫn lộn thì phần đệm bị tính như token thật.
    # Xếp gần nhau -> ít đệm -> nhanh hơn thấy rõ, không đổi kết quả một chút nào.
    con.sort(key=lambda i: len(cap[i][1]))

    # thay_offline=True cũng tự bật device_map="auto": trải thầy qua MỌI GPU.
    # Trên T4 x2 là 29GB thay vì 14,5GB. Đây là chỗ đã OOM lượt 25/08.
    m = load_qwen_reranker(THAY, device="cuda", load_4bit=True,
                           thay_offline=True,        # <-- cửa hợp lệ, xem docstring
                           batch_size=8, max_length=1024)
    T0 = time.time(); B = 8
    for b0 in range(0, len(con), B):
        lo = con[b0:b0+B]
        for (q, d), s in zip([chiso[i] for i in lo], m.predict([cap[i] for i in lo])):
            nhan.setdefault(q, {})[d] = float(s)
        n = b0 + len(lo)
        if (b0 // B) % 25 == 0 or n == len(con):
            json.dump(nhan, open(P_OUT, "w", encoding="utf-8"), ensure_ascii=False)
            el = time.time() - T0
            print(f"  {n:,}/{len(con):,} · {el/60:.0f} phút · {el/max(n,1):.2f} s/cặp · "
                  f"còn ~{(len(con)-n)*el/max(n,1)/3600:.1f}h", flush=True)
        if time.time() - T0 > GIO_TRAN * 3600:
            json.dump(nhan, open(P_OUT, "w", encoding="utf-8"), ensure_ascii=False)
            print(f"\n!! Chạm trần {GIO_TRAN}h. Đã lưu {sum(len(v) for v in nhan.values()):,} cặp.")
            print("   Tải outputs/ về, upload lên dataset, chạy lại -> chạy TIẾP.")
            break
json.dump(nhan, open(P_OUT, "w", encoding="utf-8"), ensure_ascii=False)




In [ ]:
# ===== Bước 4: kiểm nhãn + báo cáo =====
du = [q for q, e in nhan.items() if len(e) == K_UNGVIEN]
print(f"{len(du)}/{len(nhom)} câu có ĐỦ {K_UNGVIEN} nhãn\n")

if du:
    # Thầy có thật sự phân biệt được không? Nếu R@1 của thầy trên chính nhóm này mà thấp
    # thì nhãn là rác, đừng mang đi huấn luyện.
    h = sum(nhom[q][0] == max(nhan[q], key=nhan[q].get) for q in du) / len(du)
    import statistics as st
    tv = [nhan[q][nhom[q][0]] for q in du]
    am = [v for q in du for d, v in nhan[q].items() if d != nhom[q][0]]
    print(f"thầy xếp gold hạng 1: {h:.3f}  ({sum(nhom[q][0]==max(nhan[q],key=nhan[q].get) for q in du)}/{len(du)})")
    print(f"điểm gold   : trung vị {st.median(tv):.4f}")
    print(f"điểm negative: trung vị {st.median(am):.4f}")
    print(f"\nCỔNG: thầy phải xếp gold hạng 1 ở >= 0,60 nhóm. "
          f"{'✅ ĐẠT' if h >= 0.60 else '❌ KHÔNG ĐẠT — nhãn yếu, ĐỪNG huấn luyện, báo lại'}")

json.dump({"n_cau": len(nhom), "k": K_UNGVIEN, "seed": SEED, "exc": EXC,
           "thay": THAY, "du_nhan": len(du)},
          open(f"{OUT}/meta_nhan_thay.json", "w", encoding="utf-8"))

# CHỐT AN TOÀN: notebook sinh nhãn TUYỆT ĐỐI không được đẻ ra bài nộp.
xau = [p for p in glob.glob("/kaggle/working/**/*", recursive=True) if "submission" in os.path.basename(p)]
assert not xau, f"⛔ notebook thầy KHÔNG ĐƯỢC tạo file submission, thấy: {xau}"
print("\n✅ không có file submission nào — đúng luật. TẢI outputs/ VỀ TRƯỚC KHI ĐÓNG PHIÊN.")

